# 🎙️ VocalMind — GPU Inference Server
### Run Cell 1 once → it auto-restarts kernel → then run Cells 2, 3, 4

## ⚙️ Cell 1 — Install Dependencies
> Run once. Kernel restarts automatically. After restart, skip to Cell 2.

In [1]:
import importlib, IPython

try:
    [importlib.import_module(p) for p in ("funasr", "pyannote.audio", "whisper", "uvicorn")]
    print("✅ All packages installed — skip to Cell 2.")
except ImportError:
    print("📦 Installing packages (takes ~3 min)...")
    
    %pip install -q llvmlite kaldiio torch-complex hydra-core jaconv jamo tensorboardX modelscope librosa soundfile openai-whisper nest-asyncio uvicorn fastapi pyngrok python-multipart asteroid-filterbanks einops huggingface_hub lightning torchmetrics speechbrain==0.5.16 torch-audiomentations
    %pip install -q funasr pyannote.audio pyannote.core pyannote.database pyannote.metrics pyannote.pipeline
    %pip install -q "numpy<2.0" scipy==1.14.1 numba
    
    print("\n✅ Done — restarting kernel...")
    IPython.Application.instance().kernel.do_shutdown(restart=True)

📦 Installing packages (takes ~3 min)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.6/630.6 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s 

## 🔑 Cell 2 — Config
> **Change these values every new session if tokens expire. HF_TOKEN never expires.**

In [ ]:
HF_TOKEN       = "NA"           # never changes
NGROK_TOKEN    = "NA"         # never changes
NGROK_DOMAIN   = "etta-cleistogamous-untangentially.ngrok-free.dev"        # e.g. imani-xxx.ngrok-free.dev — never changes
BACKEND_URL    = "https://archidiaconal-ebonie-grimly.ngrok-free.dev/internal/set-kaggle-url"
print("✅ Config loaded")

✅ Config loaded


## 🧠 Cell 3 — Load Models
> Takes ~2 min first run, ~30s after (cached on disk).

In [ ]:
import sys, os, tempfile, torch, types as _t
HF_TOKEN       = "NA"           # never changes
NGROK_TOKEN    = "NA"         # never changes
NGROK_DOMAIN   = "etta-cleistogamous-untangentially.ngrok-free.dev"        # e.g. imani-xxx.ngrok-free.dev — never changes
BACKEND_URL    = "https://archidiaconal-ebonie-grimly.ngrok-free.dev/internal/set-kaggle-url"
# ── torchaudio patch ──
# import torchaudio
# if not hasattr(torchaudio, "list_audio_backends"):
#     torchaudio.list_audio_backends = lambda: ["sox", "soundfile"]

from funasr import AutoModel
import whisper
from pyannote.audio import Pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")

print("[1/4] Loading Emotion Model (base) on CPU...")
emotion_model = AutoModel(model="iic/emotion2vec_plus_base", hub="ms", disable_update=True, device="cpu")
print("      ✅ Done")

print("[2/4] Loading Diarization Pipeline...")
diarization_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN)
diarization_pipeline.to(torch.device(device))
print("      ✅ Done")

print("[3/4] Loading Whisper Medium...")
whisper_model = whisper.load_model("medium", device=device)
print("      ✅ Done")

print("[4/4] Loading Silero VAD...")
vad_model, vad_utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', force_reload=False)
(get_speech_timestamps, _, read_audio, *_) = vad_utils
print("      ✅ Done\n")
# diarization_pipeline.hyperparameters["clustering"]["threshold"] = 0.30

In [ ]:
# # DEBUG — run once to inspect diarization output
# import torchaudio, tempfile, os
# from pathlib import Path

# # find any wav file
# wavs = list(Path("/kaggle/input").glob("**/*.wav")) + list(Path("/kaggle/working").glob("**/*.wav"))
# print("Found wavs:", wavs[:3])

# if wavs:
#     waveform, sr = torchaudio.load(str(wavs[0]))
#     if sr != 16000:
#         waveform = torchaudio.functional.resample(waveform, sr, 16000)
#     with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
#         torchaudio.save(f.name, waveform, 16000)
#         d = diarization_pipeline(f.name)
#         os.remove(f.name)

#     print("type:", type(d))
#     print("dir:", [x for x in dir(d) if not x.startswith("_")])
#     print("repr:", repr(d)[:1000])

## 🚀 Cell 4 — Start Server
> Runs forever. Stop only to restart Kaggle session.

In [ ]:
import os, subprocess, tempfile, asyncio
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)

import nest_asyncio, threading, time, requests as req, uvicorn, torch
from fastapi import FastAPI, File, UploadFile
from pyngrok import ngrok
MIN_EMOTION_DURATION_S = 0.8
nest_asyncio.apply()
time.sleep(1)

gpu_lock = asyncio.Lock()

# ── Inference ─────────────────────────────────────────────
def run_transcribe(path):
    torch.cuda.empty_cache()
    res = whisper_model.transcribe(path, language="en", fp16=torch.cuda.is_available())
    keep = ("start", "end", "text")
    clean = lambda s: {k: round(v, 2) if isinstance(v, float) else v for k, v in s.items() if k in keep}
    return {"text": res.get("text", "").strip(), "language": res.get("language", ""),
            "segments": [clean(s) for s in res.get("segments", [])]}

def run_emotion(path):
    torch.cuda.empty_cache()
    res = emotion_model.generate(path, output_dir="/tmp", granule="utterance", df_save=False)
    if not res: return {"top_emotion": "unknown", "emotions": []}
    labels, scores = res[0].get("labels", []), res[0].get("scores", [])
    emotions = sorted([{"label": l, "score": s} for l, s in zip(labels, scores)],
                       key=lambda x: x["score"], reverse=True)
    return {"top_emotion": emotions[0]["label"] if emotions else "unknown", "emotions": emotions}

def run_diarize(path):
    torch.cuda.empty_cache()
    result = diarization_pipeline(path, min_speakers=1)

    # pyannote 4.x returns DiarizeOutput, not Annotation directly
    if hasattr(result, 'speaker_diarization'):
        annotation = result.speaker_diarization
    else:
        annotation = result

    segments = []
    if hasattr(annotation, "itertracks"):
        for turn, _, speaker in annotation.itertracks(yield_label=True):
            segments.append({
                "start": round(float(turn.start), 2),
                "end": round(float(turn.end), 2),
                "speaker": speaker
            })
    return {"segments": segments}


def _cut_wav_segment(src_path, start_s, end_s, min_duration=MIN_EMOTION_DURATION_S):
    start_s, end_s = _normalize_segment_window(start_s, end_s, min_duration)
    duration = max(end_s - start_s, min_duration)
    out = tempfile.NamedTemporaryFile(delete=False, suffix=".wav").name
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start_s),
        "-i", src_path,
        "-t", str(duration),
        "-ac", "1",
        "-ar", "16000",
        "-af", f"apad=pad_dur={duration}",
        "-acodec", "pcm_s16le",
        out,
    ], capture_output=True, text=True, check=True)
    return out

def _emotion_from_path(path):
    res = emotion_model.generate(path, output_dir="/tmp", granule="utterance", df_save=False)
    if not res:
        return {"emotion": "unknown", "emotion_scores": []}

    labels = res[0].get("labels", [])
    scores = res[0].get("scores", [])
    ranked = sorted(
        [{"label": l, "score": float(s)} for l, s in zip(labels, scores)],
        key=lambda x: x["score"],
        reverse=True,
    )
    return {
        "emotion": ranked[0]["label"] if ranked else "unknown",
        "emotion_scores": ranked,
    }

def run_segment_emotions(path, segments):
    enriched = []
    for seg in segments:
        clip_path = _cut_wav_segment(path, seg["start"], seg["end"])
        try:
            emo = _emotion_from_path(clip_path)
        finally:
            os.remove(clip_path)

        enriched.append({
            **seg,
            "emotion": emo["emotion"],
            "emotion_scores": emo["emotion_scores"],
        })
    return enriched


def run_vad(path):
    wav = read_audio(path, sampling_rate=16000)
    timestamps = get_speech_timestamps(
        wav, vad_model,
        sampling_rate=16000,
        return_seconds=True
    )
    return {"speech_segments": timestamps}
        
def _normalize_segment_window(start_s, end_s, min_duration=MIN_EMOTION_DURATION_S):
    start_s = max(float(start_s or 0.0), 0.0)
    end_s = max(float(end_s or start_s), start_s)
    duration = end_s - start_s
    if duration < min_duration:
        pad = (min_duration - duration) / 2.0
        start_s = max(0.0, start_s - pad)
        end_s = start_s + min_duration
    return round(start_s, 3), round(end_s, 3)

def run_full(path):
    torch.cuda.empty_cache()
    t = run_transcribe(path)

    torch.cuda.empty_cache()
    d = run_diarize(path)

    diar_segs = d.get("segments", [])
    for seg in t["segments"]:
        mid = (seg["start"] + seg["end"]) / 2
        seg["speaker"] = next(
            (ds["speaker"] for ds in diar_segs if ds["start"] <= mid <= ds["end"]),
            "UNKNOWN",
        )

    torch.cuda.empty_cache()
    enriched_segments = run_segment_emotions(path, t["segments"])

    whole_call = run_emotion(path)

    return {
        "text": t["text"],
        "language": t["language"],
        "segments": enriched_segments,
        "top_emotion": whole_call["top_emotion"],
        "emotions": whole_call["emotions"],
    }

# ── FastAPI ───────────────────────────────────────────────
app = FastAPI()

async def _run_with_lock(fn, file: UploadFile):
    ext = os.path.splitext(file.filename or ".wav")[1] or ".wav"
    with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
        tmp.write(await file.read()); path = tmp.name
    try:
        async with gpu_lock:
            return await asyncio.to_thread(fn, path)
    finally:
        os.remove(path)

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)): return await _run_with_lock(run_transcribe, file)

@app.post("/emotion")
async def emotion(file: UploadFile = File(...)): return await _run_with_lock(run_emotion, file)

@app.post("/diarize")
async def diarize(file: UploadFile = File(...)): return await _run_with_lock(run_diarize, file)

@app.post("/vad")
async def vad(file: UploadFile = File(...)): return await _run_with_lock(run_vad, file)

@app.post("/full")
async def full(file: UploadFile = File(...)): return await _run_with_lock(run_full, file)

# ── Start ─────────────────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
url = ngrok.connect(8000, domain=NGROK_DOMAIN).public_url

print(f"\n{'='*57}")
print(f"🚀 VOCALMIND INFERENCE SERVER:\n   {url}")
print(f"{'='*57}")
print(f"  POST {url}/transcribe  → Whisper STT")
print(f"  POST {url}/emotion     → Emotion")
print(f"  POST {url}/diarize     → Diarization")
print(f"  POST {url}/vad         → Voice Activity Detection")
print(f"  POST {url}/full        → All combined")
print(f"  GET  {url}/health      → Health check")
print(f"{'='*57}\n")

def notify():
    try:
        r = req.post(BACKEND_URL, json={"kaggle_url": url}, timeout=5)
        print(f"✅ Backend notified: {r.status_code}")
    except Exception as e:
        print(f"⚠️  Backend not reachable — set manually: {url}")
threading.Thread(target=notify, daemon=True).start()

threading.Thread(target=uvicorn.run, args=(app,),
                 kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "error"},
                 daemon=True).start()

# try:
#     while True: time.sleep(1)
# except KeyboardInterrupt:
#     ngrok.kill()

## 📊🎯 Evaluation - vox_audio

In [ ]:
%%time
import os, zipfile, urllib.request, glob, json, time, requests, sys, subprocess, tempfile
from tqdm import tqdm
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.diarization import DiarizationErrorRate
import soundfile as sf

SERVER = "http://localhost:8000"
SUBSET_SIZE = 100
MAX_DUR = 240.0  # Prevent Kaggle OOM

# 1. Download & Extract
if not os.path.exists("vox_audio"):
    urllib.request.urlretrieve("https://www.robots.ox.ac.uk/~vgg/data/voxconverse/data/voxconverse_test_wav.zip", "audio.zip")
    urllib.request.urlretrieve("https://github.com/joonson/voxconverse/archive/refs/heads/master.zip", "rttm.zip")
    with zipfile.ZipFile("audio.zip", 'r') as z: z.extractall("vox_audio")
    with zipfile.ZipFile("rttm.zip", 'r') as z: z.extractall("vox_rttm")

wav_files = sorted(glob.glob("vox_audio/voxconverse_test_wav/*.wav"))[:SUBSET_SIZE]
metric = DiarizationErrorRate()
results = []

# 2. Evaluate Loop
for wav_path in tqdm(wav_files, desc="Evaluating Pure DER"):
    file_id = os.path.basename(wav_path).replace(".wav", "")
    rttm_path = f"vox_rttm/voxconverse-master/test/{file_id}.rttm"
    
    gt = Annotation(uri=file_id)
    with open(rttm_path, "r") as f:
        for line in f:
            p = line.strip().split()
            gt[Segment(float(p[3]), float(p[3]) + float(p[4]))] = p[7]
            
    data, sr = sf.read(wav_path)
    eval_dur = min(MAX_DUR, len(data) / sr)
    
    tmp_wav = tempfile.mktemp(suffix=".wav")
    sf.write(tmp_wav, data[:int(eval_dur * sr)], sr)

    try:
        t0 = time.time()
        with open(tmp_wav, "rb") as f:
            res = requests.post(f"{SERVER}/diarize", files={"file": f}, timeout=300).json()
        dt = time.time() - t0
        
        # Safeguard: Flag empty responses immediately
        if not res.get("segments"):
            print(f"\n⚠️ Warning: Server returned empty segments for {file_id}. Raw response: {res}")
            
    except Exception as e:
        print(f"\n⚠️ Failed {file_id}: {e}")
        continue
    finally:
        if os.path.exists(tmp_wav): os.remove(tmp_wav)
        
    hyp = Annotation(uri=file_id)
    for seg in res.get("segments", []):
        hyp[Segment(seg["start"], seg["end"])] = seg.get("speaker", "UNKNOWN")
        
    # Strictly bound evaluation to the exact cropped audio duration
    uem = Timeline([Segment(0.0, eval_dur)])
    
    detail = metric(gt, hyp, uem=uem, detailed=True)
    total = detail["total"] if detail["total"] > 0 else 1.0 
    
    results.append({
        "id": file_id, 
        "der": round(detail["diarization error rate"], 4),
        "miss": round(detail["missed detection"] / total, 4),
        "fa": round(detail["false alarm"] / total, 4),
        "conf": round(detail["confusion"] / total, 4),
        "time": round(dt, 1)
    })

# 3. Final Report
if not results:
    print("\n❌ Pipeline failed entirely. No results to report.")
    sys.exit()

global_detail = metric[:]
overall_der = abs(metric)
g_total = global_detail["total"] if global_detail["total"] > 0 else 1.0

print(f"\n📊 PURE DIARIZATION EVAL | OVERALL DER: {overall_der:.4f}")
print(f"  ├ Missed Speech:  {global_detail['missed detection'] / g_total:.4f}")
print(f"  ├ False Alarms:   {global_detail['false alarm'] / g_total:.4f}")
print(f"  └ Speaker Conf:   {global_detail['confusion'] / g_total:.4f}")

report = {"dataset": "VoxConverse", "overall_der": round(overall_der, 4), "samples": results}
with open("voxconverse_pure_diarization_report.json", "w") as f: json.dump(report, f, indent=2)
print("💾 Saved: voxconverse_pure_diarization_report.json")

In [ ]:
from pprint import pprint
with open("/kaggle/working/voxconverse_pure_diarization_report.json", 'r') as file: 
    print(file.read())
    pprint(res)